In [1]:
import cProfile
import numpy as np
import pandas as pd
from recovery_model.recovery_model_refactored import RecoveryModel

pd.set_option("multi_sparse", False)

In [2]:
# Select a folder for the data to be used
folder = "Toy_WEEE_v2"  # test_1  test_2  Toy_WEEE_v2
layer_0 = "flow"
layer_1 = "product"
layer_2 = "component"
layer_3 = "material"
layer_4 = "element"

In [3]:
layer_names = (layer_0, layer_1, layer_2, layer_3, layer_4)
all_symbols = {layer_1: "P*", layer_2: "C*", layer_3: "M*", layer_4: "E*"}

metadata = {
    "path": f"data/{folder}/",
    "filename": "metadata.csv",
}
composition = {
    "path": f"data/{folder}/",
    "filename": "composition.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Layer 1": layer_1,
        "Layer 2": layer_2,
        "Layer 3": layer_3,
        "Layer 4": layer_4,
        "Value": "data",
        "Year": "year",
        "Scenario": "scenario",
        "Location": "region",
        "parameterCode": "parameterCode",
        "UoM": "unit",
    },
    "parameterCode": {
        layer_2: "c-p",
        layer_3: "m-c",
        layer_4: "e-m",
    },
}

inputs = {
    "path": f"data/{folder}/",
    "filename": "inputs.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Substance_main_parent": layer_1,
        "Value": "data",
        "Year": "year",
        "Unit": "unit",
    },
}

tcs = {
    "path": f"data/{folder}/",
    "filename": "TCs.csv",
    "mapper": {
        # inflows / outflows
        "input_flow": "inflows",
        "output_flow": "outflows",
        # levels
        "input_layer": "input_layer1_level",
        "input_sub_layer": "input_layer2_level",
        "output_layer": "output_layer1_level",
        # keys
        "input_layer_key": "input_layer1_key",
        "input_sub_layer_key": "input_layer2_key",
        "output_target_key": "output_layer1_key",
        # other columns
        "value": "data",
        "Year": "year",
    },
    "all_symbols": {
        layer_1: "P*",
        layer_2: "C*",
        layer_3: "M*",
        layer_4: "E*",
    },
}

---


# Recovery model


In [4]:
weee = RecoveryModel(
    metadata=metadata,
    composition=composition,
    inputs=inputs,
    tcs=tcs,
    layer_names=layer_names,
    save_intermediary_steps=False,
    save_duplicates=False,
)

---

# Consolidation


In [5]:
file = f"results/duplicates.csv"
duplicates = pd.read_csv(file, header=[0, 1], index_col=0).drop(columns=[("info", "process"), ("info", "technology")])
duplicates.head()

,info,outflow,outflow,outflow,outflow,outflow,inflow,inflow,inflow,inflow,inflow,info,info
,original row,flow,product,component,material,element,flow,product,component,material,element,data,priority
538,538,1,4,26,11,6,16,4,26,11,6,0.4,2111
538,538,1,4,26,19,6,16,4,26,19,6,0.4,2111
538,538,1,4,26,25,6,16,4,26,25,6,0.4,2111
538,538,1,4,26,24,6,16,4,26,24,6,0.4,2111
538,538,1,4,26,23,6,16,4,26,23,6,0.4,2111


In [6]:
# temporary rename columns for easier processing
cols = list(duplicates.columns)
duplicates.columns = list(range(len(duplicates.columns)))
duplicates[0] = duplicates[0] + 2  # for exact match in excel

# get what the conflict values (for tc) are
s = duplicates.groupby(list(range(1, 11)))[11].unique().to_frame()
s[("info", "values_count")] = s[11].map(len)

# get what the conflict priorities (for tc) are
s2 = duplicates.groupby(list(range(1, 11)))[12].unique().to_frame()
s2[("info", "priority_count")] = s2[12].map(len)
s = pd.concat([s, s2], axis=1)
del s2

# get what original rows are concerned (in the tcs csv file)
s2 = duplicates.groupby(list(range(1, 11)))[0].unique().to_frame()
s = pd.concat([s, s2], axis=1)
del s2

# rename
s = s.reset_index().rename(columns={i: v for i, v in enumerate(cols)})
s.columns = pd.MultiIndex.from_tuples(s.columns)
s = weee.decode_label(s)
s.to_csv("consolidation/tcs_collision.csv")
s

,outflow,outflow,outflow,outflow,outflow,inflow,inflow,inflow,inflow,inflow,info,info,info,info,info
,flow,product,component,material,element,flow,product,component,material,element,data,values_count,priority,priority_count,original row
0,WEEE_2RM_thermal2Smelter_other,NaN,NaN,NaN,Ag,WEEE_chem_thermal,NaN,NaN,NaN,Ag,"[0.59, 0.18]",2,[2111],1,"[541, 538]"
1,WEEE_2RM_thermal2Smelter_other,NaN,NaN,NaN,Al,WEEE_chem_thermal,NaN,NaN,NaN,Al,"[0.53, 0.06]",2,[2111],1,"[542, 544]"
2,WEEE_2RM_thermal2Smelter_other,NaN,NaN,NaN,Cu,WEEE_chem_thermal,NaN,NaN,NaN,Cu,"[0.4, 0.18]",2,[2111],1,"[540, 545]"
3,WEEE_2RM_thermal2Smelter_other,NaN,NaN,AlAndAlAlloys,Ag,WEEE_chem_thermal,NaN,NaN,AlAndAlAlloys,Ag,"[0.59, 0.18]",2,[2111],1,"[541, 538]"
4,WEEE_2RM_thermal2Smelter_other,NaN,NaN,AlAndAlAlloys,Al,WEEE_chem_thermal,NaN,NaN,AlAndAlAlloys,Al,"[0.53, 0.06]",2,[2111],1,"[542, 544]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124057,WEEE_generatedComplementaryMetalScrap,WEEE_Cat1,ComponentShadowWEEE,wood,Pd,WEEE_generatedDedicated,WEEE_Cat1,ComponentShadowWEEE,wood,Pd,[0.21],1,"[1122, 1112]",2,"[546, 2]"
124058,WEEE_generatedComplementaryMetalScrap,WEEE_Cat1,ComponentShadowWEEE,wood,Sc,WEEE_generatedDedicated,WEEE_Cat1,ComponentShadowWEEE,wood,Sc,[0.21],1,"[1122, 1112]",2,"[546, 2]"
124059,WEEE_generatedComplementaryMetalScrap,WEEE_Cat1,ComponentShadowWEEE,wood,Sm,WEEE_generatedDedicated,WEEE_Cat1,ComponentShadowWEEE,wood,Sm,[0.21],1,"[1122, 1112]",2,"[546, 2]"


In [10]:
del s

---

# Metadata


In [11]:
weee.dims

(19, 8, 83, 31, 18)

In [12]:
weee.size

7039728

In [13]:
print(weee)

{   'Layer 1': {   'P*': 7,
                   'WEEE_Cat1': 0,
                   'WEEE_Cat2': 1,
                   'WEEE_Cat3': 2,
                   'WEEE_Cat4a': 3,
                   'WEEE_Cat4b': 4,
                   'WEEE_Cat5': 5,
                   'WEEE_Cat6': 6},
    'Layer 2': {   'C*': 82,
                   'ComponentShadowWEEE': 0,
                   'ConductivitySensor': 1,
                   'LED': 2,
                   'LidarSensor (&lt;15cm)': 3,
                   'MagnetSensor': 4,
                   'PCBWEEECat': 5,
                   'PCBunspecified': 6,
                   'PVBacksheet': 7,
                   'PVEncapsulationEVA': 8,
                   'PVFrameAl': 9,
                   'PVSolarGlass': 10,
                   'PVcellSi': 11,
                   'WEEEEvaporator': 12,
                   'brush': 13,
                   'cableUnspecified': 14,
                   'cableWOPlug': 15,
                   'cableWPlug': 16,
                   'cableWithConne

---

# Linear equations


In [5]:
weee.lneqs

<7039728x7039728 sparse matrix of type '<class 'numpy.float64'>'
	with 3309319 stored elements in Compressed Sparse Row format>

---

# Constant terms


In [6]:
weee.y

<7039728x1 sparse array of type '<class 'numpy.int64'>'
	with 7 stored elements in Compressed Sparse Column format>

---

# Solver


In [7]:
solution = weee.solve(expand=False)
solution

,flow,product,component,material,element,mass
0,WEEE_generatedComplementaryExported,WEEE_Cat1,NaN,NaN,NaN,220253.788618
1,WEEE_generatedComplementaryExported,WEEE_Cat1,ComponentShadowWEEE,NaN,NaN,43463.414287
2,WEEE_generatedComplementaryExported,WEEE_Cat1,ComponentShadowWEEE,NaN,Ag,26078.048572
3,WEEE_generatedComplementaryExported,WEEE_Cat1,ComponentShadowWEEE,AlAndAlAlloys,NaN,21.775171
4,WEEE_generatedComplementaryExported,WEEE_Cat1,ComponentShadowWEEE,AlAndAlAlloys,Ag,0.011367
...,...,...,...,...,...,...
84160,WEEE_categ_mechRec1,WEEE_Cat6,wheel,polymerPA6GF30,Ag,110.805705
84161,WEEE_categ_mechRec1,WEEE_Cat6,wheel,polymerPPTF20,NaN,202.940851
84162,WEEE_categ_mechRec1,WEEE_Cat6,wheel,polymerPPTF20,Ag,110.805705
84163,WEEE_categ_mechRec1,WEEE_Cat6,wheel,polymerPPTF30,NaN,65.581221


---

# Performance


In [8]:
cProfile.run(
    "RecoveryModel(metadata, composition, inputs, tcs, layer_names, False, False)",
    "results/performances/RecoveryModel.pstats",
)

cProfile.run(
    "weee.solve(expand=False)",
    "results/performances/Solver.pstats",
)

# # ! then go into results/performances/ and run the following commands
# # (-n 2 means cutoff at 2%)
# gprof2dot -f pstats -n 1 RecoveryModel.pstats | dot -Tpng -o RecoveryModel.png
# gprof2dot -f pstats -n 1 Solver.pstats | dot -Tpng -o Solver.png

In [9]:
cProfile.run(
    "RecoveryModel(metadata, composition, inputs, tcs, layer_names, False, False)",
    "results/performances/RecoveryModel.prof",
)

cProfile.run(
    "weee.solve(expand=False)",
    "results/performances/Solver.prof",
)

# # ! then go into results/performances/ and run the following commands
# snakeviz RecoveryModel.prof
# snakeviz Solver.prof